# Module 37: Cluster Deployment Modes & Production spark-submit Mastery

## 1. What It Is
Apache Spark applications can be submitted to various cluster managers (Standalone, Kubernetes, YARN, Local) using the unified **`spark-submit`** command-line utility.
The execution topology is governed by **Deploy Mode**:
- **Client Mode**: The Driver program runs directly on the machine where `spark-submit` was launched (great for interactive notebooks & debugging).
- **Cluster Mode**: The Driver program runs inside a worker node container managed by the cluster orchestrator (essential for automated scheduled production jobs).

## 2. Why It Matters
Submitting an application with the wrong deploy mode or poorly sized executor flags causes driver disconnection failures, network saturation, and wasteful resource allocation.
Production engineering requires mastering `spark-submit` flags for memory overhead, dependency distribution (`--py-files`), and dynamic resource allocation.

## 3. Concept Diagram
```
  Client Deploy Mode (Debugging / Interactive):
  [ Gateway Machine (Laptop / CI/CD) ]
  └── Driver Process (JVM + Python) <──Network RPC──► [ Worker Nodes: Executors ]
  * If gateway disconnects, the job DIES! *

  Cluster Deploy Mode (Production Automated):
  [ Gateway Machine ] ──► Submits Job Spec ──► [ Cluster Manager (K8s / YARN / Standalone) ]
  (Gateway can shut down immediately!)              │
                                                    ├──► Node 1: [ Driver Container ]
                                                    ├──► Node 2: [ Executor Container ]
                                                    └──► Node 3: [ Executor Container ]
```

## 7. Common Mistakes & Gotchas
- **Using Client Mode in Production Airflow / Cron**: If your Airflow worker restarts or network drops, all running Spark client-mode jobs immediately terminate. Use `--deploy-mode cluster` for scheduled batch jobs.
- **Forgetting `--py-files` for custom Python modules**: If your Python script imports helper functions from a local directory (e.g. `import my_utils`), you must zip the module and pass `--py-files my_utils.zip` so worker nodes receive the Python bytecode.
- **Specifying local file paths in Cluster Mode**: In cluster mode, the driver runs on an arbitrary remote worker node. Passing `--files /home/user/config.json` fails because that path only exists on your local laptop! Upload files to S3/HDFS or distribute via `--files`.

## 8. Official Documentation
- Submitting Applications (`spark-submit`): https://spark.apache.org/docs/latest/submitting-applications.html
- Spark on Kubernetes: https://spark.apache.org/docs/latest/running-on-kubernetes.html

In [ ]:
import sys
import os

# Project root setup (compatible with both standalone execution and Jupyter notebooks)
if "__file__" in locals():
    project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../..") if "notebooks" in os.getcwd() else os.getcwd())
    os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)
from scripts.spark_session import get_spark_session

def validate_and_generate_submit(
    app_file: str,
    cluster_manager: str,
    deploy_mode: str,
    num_executors: int,
    executor_cores: int,
    executor_memory_gb: int,
    extra_packages: list = None
) -> str:

Validates cluster sizing rules and generates a production-grade spark-submit command.